In [ ]:
#| default_exp loss

# Loss functions

> Im lost too. Phds are dumb.

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import torch, numpy as np, torch.nn.functional as F, torch.nn as nn, collections, copy
from typing import Callable

import pytorch_metric_learning.losses as losses
from torchsurv.loss.cox import neg_partial_log_likelihood as neg_partial_log_likelihood_cox
#from torchsurv.loss.weibull import neg_log_likelihood as neg_log_likelihood_weibull

In [ ]:
#| export
class NTXentLoss_poly(nn.Module):
    """
    Normalized temperature scaled cross entropy loss.    
    """
    def __init__(self, batch_size, temperature=0.2, polynomial_expanison=True):
        super(NTXentLoss_poly, self).__init__()
        self.batch_size = batch_size
        self.temperature = temperature
        self.polynomial_expanison = polynomial_expanison
        self.softmax = nn.Softmax(dim=-1)
        self.mask_samples_from_same_repr = self._get_correlated_mask().type(torch.bool)
        self.criterion = nn.CrossEntropyLoss(reduction="mean")


    def _get_correlated_mask(self):
        diag = np.eye(2 * self.batch_size)
        l1 = np.eye((2 * self.batch_size), 2 * self.batch_size, k=-self.batch_size)
        l2 = np.eye((2 * self.batch_size), 2 * self.batch_size, k=self.batch_size)
        mask = torch.from_numpy((diag + l1 + l2))
        mask = (1 - mask).type(torch.bool)
        return mask

    def forward(self, zis, zjs):
        representations = torch.cat([zjs, zis], dim=0)
        #similarity_matrix = F.cosine_similarity(representations.unsqueeze(1), representations.unsqueeze(0), dim=-1)
        ## below is more memory saving
        similarity_scores = []
        for i in range(representations.shape[0]):
            for j in range(representations.shape[0]):
                # Calculate the cosine similarity for the pair
                similarity = F.cosine_similarity(representations[i], representations[j], dim=-1)
                # Append the similarity score to the list
                similarity_scores.append(similarity.item())
        similarity_matrix = torch.as_tensor(similarity_scores, device=representations.device).reshape(2 * self.batch_size, 2 * self.batch_size)
        # filter out the scores from the positive samples
        l_pos = torch.diag(similarity_matrix, self.batch_size)
        r_pos = torch.diag(similarity_matrix, -self.batch_size)
        positives = torch.cat([l_pos, r_pos]).view(2 * self.batch_size, 1)

        negatives = similarity_matrix[self.mask_samples_from_same_repr].view(2 * self.batch_size, -1)

        logits = torch.cat((positives, negatives), dim=1)
        logits /= self.temperature

        """Criterion has an internal one-hot function. Here, make all positives as 1 while all negatives as 0. """
        labels = torch.zeros(2 * self.batch_size).long().to(zis.device)
        CE = self.criterion(logits, labels)
        
        # Add poly loss
        if self.polynomial_expanison:
            onehot_label = torch.cat((torch.ones(2 * self.batch_size, 1),torch.zeros(2 * self.batch_size, negatives.shape[-1])),dim=-1).long().to(zis.device)
            pt = torch.mean(onehot_label* F.softmax(logits,dim=-1))
            epsilon = self.batch_size # not entirely sure what this should be. TFC had it as batch size. 
            loss = CE + epsilon * (1/self.batch_size - pt) # polyloss = https://blog.ceshine.net/post/polyloss/#how-and-why-does-it-work
        else:
            loss = CE
        return loss
    
class NTXentLoss(nn.Module):
    """
    Normalized temperature scaled cross entropy loss. This is far more efficient than the above. No polynomial expansion though.  
    """
    def __init__(self, temperature=0.2):
        super(NTXentLoss, self).__init__()
        self.temperature = temperature
        self.loss = losses.SelfSupervisedLoss(losses.NTXentLoss(temperature=temperature))
    
    def forward(self, zis, zjs):
        if zis.is_nested:
            zis = zis.to_padded_tensor(padding=0)
            zjs = zjs.to_padded_tensor(padding=0)
            zis = zis.flatten(start_dim=-3)
            zjs = zjs.flatten(start_dim=-3)
        return self.loss(zis, zjs)

In [ ]:
#| notest
x1, x2 = torch.randn(2,4,100,100), torch.randn(2,4,100,100)

l = NTXentLoss(temperature=0.5)
l(x1.flatten(start_dim=-3),x2.flatten(start_dim=-3)), l(x1.permute(0,1,3,2).flatten(start_dim=-3), x2.permute(0,1,3,2).flatten(start_dim=-3))

(tensor(1.0870), tensor(1.0870))

In [ ]:
#| notest
#| notest
batch_size = 2
n_vars = 7
max_len = 480
patch_len = 100

# Create sequences of different lengths
seq_lens = torch.randint(50, max_len, (batch_size,))

# Create input tensors with different sequence lengths
x_list = [torch.randn(length, n_vars, patch_len) for length in seq_lens]
x_list2 = [torch.randn(length, n_vars, patch_len) for length in seq_lens]
x_nested = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)
x_nested2 = torch.nested.as_nested_tensor(x_list2, layout=torch.jagged)

loss = NTXentLoss(temperature=0.5)
loss(x_nested, x_nested2)

tensor(1.0981)

In [ ]:
#| export
def mse_loss(preds, target):
    """
    preds:   [bs x num_patch x n_vars x patch_len]
    targets: [bs x num_patch x n_vars x patch_len] 
    """
    if preds.is_nested:
        loss = 0
        for pred, targ in zip(preds.unbind(), target.unbind()):
            loss += F.mse_loss(pred, targ, reduction='mean')
        return loss / preds.size(0)
    else:
        return F.mse_loss(preds, target, reduction='mean')

def mae_loss(preds, target):
    """
    preds:   [bs x num_patch x n_vars x patch_len]
    targets: [bs x num_patch x n_vars x patch_len] 
    """
    if preds.is_nested:
        loss = 0
        for pred, targ in zip(preds.unbind(), target.unbind()):
            loss += F.l1_loss(pred, targ, reduction='mean')
        return loss / preds.size(0)
    else:
        return F.l1_loss(preds, target, reduction='mean')


def mape(preds, target):
    epsilon = np.finfo(np.float64).eps # from sklearn
    if preds.is_nested:
        loss = 0
        for pred, targ in zip(preds.unbind(), target.unbind()):
            loss += (((targ - pred).mean(dim=-1).abs()) / (targ.mean(dim=-1).abs()).clamp(min=epsilon)).sum()
        return loss / preds.size(0)
    else:
        return (((target - preds).mean(dim=-1).abs()) / (target.mean(dim=-1).abs()).clƒamp(min=epsilon)).sum()
    

def cosine_similarity_loss(preds,target):
    """
    preds:   [bs x num_patch x n_vars x patch_len]
    targets: [bs x num_patch x n_vars x patch_len] 
    """
    if preds.is_nested:
        loss = 0
        for pred, targ in zip(preds.unbind(), target.unbind()):
            sim = F.cosine_similarity(pred, targ, dim=-1)
            loss += sim.sum() / torch.as_tensor(torch.numel(pred), device=preds.device)
        return -loss / preds.size(0)
    else:
        sim = F.cosine_similarity(preds,target, dim=-1)
        n_elements = torch.numel(preds)
        sim = sim.sum() / torch.as_tensor(n_elements, device=preds.device)
        return -sim

def huber_loss(preds, target, delta=1):
    """
    preds:   [bs x num_patch x n_vars x patch_len]
    targets: [bs x num_patch x n_vars x patch_len] 
    """
    if preds.is_nested:
        # Calculate loss for each item in the batch separately
        total_loss = 0
        for pred, targ in zip(preds.unbind(), target.unbind()):
            # Calculate huber loss with reduction='none' for this item
            total_loss += F.huber_loss(pred, targ, delta=delta, reduction='mean')
            
        # Calculate average loss across all actual elements
        return total_loss / preds.size(0)
    else:   
        return F.huber_loss(preds, target, delta=delta, reduction='mean')
    

def smoothl1_loss(preds, target):
    if preds.is_nested:
        loss = 0
        for pred, targ in zip(preds.unbind(), target.unbind()):
            loss += F.smooth_l1_loss(pred, targ, reduction='mean')
        return loss / preds.size(0)
    else:
        return F.smooth_l1_loss(preds, target, reduction='mean')

def mse_variance_loss(preds, target, representations, alpha = 0.2):
    """
    preds:   [bs x num_patch x n_vars x patch_len]
    targets: [bs x num_patch x n_vars x patch_len] 
    representations: [bs x nvars x d_model x num_patch]
    """
    loss = 0.0
    for pred_i, target_i, representations_i in zip(preds, target, representations):
        loss = loss + F.mse_loss(pred_i, target_i, reduction='mean')
        loss = loss + alpha * F.relu(1.0 - representations_i.std(dim=-1)).mean()
    loss /= len(preds)
    return loss


In [ ]:
#| notest
x = torch.randn(4,300, 7, 100)
y = torch.randn(4,300, 7, 100)

huber_loss(x,y)

tensor(0.7208)

In [ ]:
#| notest
batch_size = 2
n_vars = 7
max_len = 480
patch_len = 100

# Create sequences of different lengths
seq_lens = torch.randint(50, max_len, (batch_size,))

# Create input tensors with different sequence lengths
x_list = [torch.randn(length, n_vars, patch_len) for length in seq_lens]
x_list2 = [torch.randn(length, n_vars, patch_len) for length in seq_lens]
x_nested = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)
x_nested2 = torch.nested.as_nested_tensor(x_list2, layout=torch.jagged)

smoothl1_loss(x_nested, x_nested2)

tensor(0.7184)

In [ ]:
#| export
def patch_continuity_loss(preds):
    """
    preds: [bs x num_patch x n_vars x patch_len]
    targets: [bs x num_patch x n_vars x patch_len]
    """
    bs, num_patch, n_vars, patch_len = preds.shape
    # Calculate difference between end of each patch and start of next
    diff = preds[:, 1:, :, 0] - preds[:, :-1, :, -1]
    
    # Mean squared difference
    continuity_loss = torch.mean(diff**2)
    
    return continuity_loss

In [ ]:
#| notest
x = torch.randn(2,2, 2, 10)

patch_continuity_loss(x)

tensor(1.8017)

In [ ]:
#| export
def demographic_adversarial_loss(demo_predictions, demo_targets):
    assert demo_predictions.shape[1] == demo_targets.shape[1], "Demographic predictor and demographics size do not match."
    total_loss = 0.0
    if torch.isnan(demo_targets).all():
        return total_loss
    for i in range(demo_predictions.shape[1]):
        pred_ = demo_predictions[:, i]
        targ = demo_targets[:, i]
        if torch.isnan(targ).all():
            continue
        elif torch.isnan(targ).any():
            mask = ~torch.isnan(targ)
            pred_ = pred_[mask]
            targ = targ[mask]
        if targ.unique().numel() <= 2:
            # Binary outcome
            loss = nn.BCEWithLogitsLoss()(pred_, targ)
        else:
            # Continuous outcome
            loss = nn.MSELoss()(pred_, targ)

        total_loss += loss

    return total_loss

In [ ]:
#| export
def multilabel_cox_loss(x, event, time, n_labels, ties_method='efron', reduction="mean"):
    """
    Multilabel Cox Loss Function
    x: [bs x n_labels] "log hazard"
    event: [bs x n_labels] - 1 for event, 0 for censored
    time: [bs x n_labels] - time of event or time censored
    """
    loss = torch.tensor([0.], device=x.device, dtype=torch.float32)
    if n_labels == 1:
        x = x.unsqueeze(-1) if x.dim() == 1 else x
        event = event.unsqueeze(-1) if event.dim() == 1 else event
        time = time.unsqueeze(-1) if time.dim() == 1 else time
    for l in range(n_labels):
        loss += neg_partial_log_likelihood_cox(x[:,l], event[:,l], time[:,l], ties_method=ties_method, reduction=reduction)
    return loss

def nll_logistic_hazard(phi, events, idx_durations, reduction = 'mean'):
    """
    Adapted from https://github.com/havakv/pycox/blob/3eccdd7fd9844a060f50fdcc315659f33a2d2dc1/pycox/models/loss.py#L18
    Negative log-likelihood of the discrete time hazard parametrized model LogisticHazard [1].
    
    Arguments:
        phi {torch.tensor} -- Estimates in (-inf, inf), where hazard = sigmoid(phi).
        idx_durations {torch.tensor} -- Event times represented as indices.
        events {torch.tensor} -- Indicator of event (1.) or censoring (0.).
            Same length as 'idx_durations'.
        reduction {string} -- How to reduce the loss.
            'none': No reduction.
            'mean': Mean of tensor.
            'sum: sum.
    
    Returns:
        torch.tensor -- The negative log-likelihood.

    References:
    [1] Håvard Kvamme and Ørnulf Borgan. Continuous and Discrete-Time Survival Prediction
        with Neural Networks. arXiv preprint arXiv:1910.06724, 2019.
        https://arxiv.org/pdf/1910.06724.pdf
    """
    if phi.shape[-1] <= idx_durations.max():
        raise ValueError(f"Network output `phi` is too small for `idx_durations`."+
                         f" Need at least `phi.shape[1] = {idx_durations.max().item()+1}`,"+
                         f" but got `phi.shape[1] = {phi.shape[1]}`")
    phi = phi.float()
    events = events.float()
    #events = events.view(-1, 1)
    #idx_durations = idx_durations.view(-1, 1)

    if events.dim() == 1:
        # [bs x n_time_bins]
        events = events.view(-1, 1)
        idx_durations = idx_durations.view(-1, 1)
        y_bce = torch.zeros_like(phi).scatter(1, idx_durations, events)
        bce = F.binary_cross_entropy_with_logits(phi, y_bce, reduction='none')
        loss = bce.cumsum(1).gather(1, idx_durations).view(-1)
    elif events.dim() == 2:
        #  soft labels, just do two separate losses and sum 
        # idx durations though needs to output correct shape
        events_1 = events[:, 0].view(-1, 1)
        idx_durations_1 = idx_durations[:, 0].view(-1, 1)
        y_bce_1 = torch.zeros_like(phi).scatter(1, idx_durations_1, events_1)
        bce_1 = F.binary_cross_entropy_with_logits(phi, y_bce_1, reduction='none')
        loss_1 = bce_1.cumsum(1).gather(1, idx_durations_1).view(-1)

        events_2 = events[:, 1].view(-1, 1)
        idx_durations_2 = idx_durations[:, 1].view(-1, 1)
        y_bce_2 = torch.zeros_like(phi).scatter(1, idx_durations_2, events_2)
        bce_2 = F.binary_cross_entropy_with_logits(phi, y_bce_2, reduction='none')
        loss_2 = bce_2.cumsum(1).gather(1, idx_durations_2).view(-1)

        # Sum the losses (soft labels already contain the mixture weights)
        loss = loss_1 + loss_2

    # y_bce = torch.zeros_like(phi).scatter(1, idx_durations, events)
    # bce = F.binary_cross_entropy_with_logits(phi, y_bce, reduction='none')
    # loss = bce.cumsum(1).gather(1, idx_durations).view(-1)
    if reduction == 'none':
        return loss
    elif reduction == 'mean':
        return loss.mean()
    elif reduction == 'sum':
        return loss.sum()
    else:
        raise ValueError(f"Reduction {reduction} not implemented.")
    

# def multilabel_weibull_loss(x, event, time, n_labels, reduction="mean"):
#     """
#     Multilabel Weibull Loss Function
#     x: [bs x n_labels x 2] "log hazard"
#     event: [bs x n_labels] - 1 for event, 0 for censored
#     time: [bs x n_labels] - time of event or time censored

#     2 dimension - The first column corresponds to the log scale parameter. The second column corresponds to the log shape parameter
#     """
#     loss = torch.tensor([0.], device=x.device, dtype=torch.float32)
#     if n_labels == 1:
#         x = x.unsqueeze(-1) if x.dim() == 1 else x
#         event = event.unsqueeze(-1) if event.dim() == 1 else event
#         time = time.unsqueeze(-1) if time.dim() == 1 else time
#     for l in range(n_labels):
#         loss += neg_log_likelihood_weibull(x[:,l], event[:,l], time[:,l], reduction=reduction)
#     return loss


In [ ]:
#| export
def cox_ph_loss(x, event, time):
    # Sort event times and get corresponding indices for sorting other tensors
    time, sorted_idx = time.sort(dim=0, descending=True)
    hazards = x.gather(0, sorted_idx)
    event = event.gather(0, sorted_idx)

    log_cumulative_hazard = torch.logcumsumexp(hazards.float(), dim=0)

    # Calculate losses for all labels simultaneously
    losses = (hazards - log_cumulative_hazard) * event
    losses = -losses  # Negative for maximization

    # Average loss per label
    label_loss = losses.sum(dim=0) / (event.sum(dim=0) + 1e-9)  # Avoid division by zero

    # Average across labels
    total_loss = label_loss.mean()

    return total_loss

In [ ]:
#| notest

x = torch.randn(2, 14)
event = torch.randint(0, 2, (2,14))
time = torch.randint(1, 300, (2,14))

cox_ph_loss(x, event, time), multilabel_cox_loss(x, event, time, 14)



/opt/miniconda3/envs/timeflies/lib/python3.12/site-packages/torchsurv/loss/cox.py:124: UserWarning: No events OR single sample. Returning zero loss for the batch
  warnings.warn("No events OR single sample. Returning zero loss for the batch")


(tensor(0.3138), tensor([3.1433], grad_fn=<AddBackward0>))

In [ ]:
#| notest

x = torch.randn(2, 20).float()
event = torch.randint(0, 2, (2,)).float()
time_idx = torch.randint(0, 18, (2,))

events = event.view(-1, 1)
idx_durations = time_idx.view(-1, 1)
y_bce = torch.zeros_like(x).scatter(1, idx_durations, events)
bce = F.binary_cross_entropy_with_logits(x, y_bce, reduction='none')
loss = bce.cumsum(1).gather(1, idx_durations).view(-1)

nll_logistic_hazard(x, event, time_idx)



tensor(7.4561)

In [ ]:
#| notest

x = torch.randn(4, 15).float()
event = torch.rand(4,2)
time_idx_1 = torch.randint(0, 15, (4,))
time_idx_2 = torch.randint(0, 15, (4,))
time_idx = torch.stack([time_idx_1, time_idx_2], dim=1)

events_1 = event[:, 0].view(-1, 1)
idx_durations_1 = time_idx[:, 0].view(-1, 1)
y_bce_1 = torch.zeros_like(x).scatter(1, idx_durations_1, events_1)
bce_1 = F.binary_cross_entropy_with_logits(x, y_bce_1, reduction='none')
loss_1 = bce_1.cumsum(1).gather(1, idx_durations_1).view(-1)

events_2 = event[:, 1].view(-1, 1)
idx_durations_2 = time_idx[:, 1].view(-1, 1)
y_bce_2 = torch.zeros_like(x).scatter(1, idx_durations_2, events_2)
bce_2 = F.binary_cross_entropy_with_logits(x, y_bce_2, reduction='none')
loss_2 = bce_2.cumsum(1).gather(1, idx_durations_2).view(-1)

# Sum the losses (soft labels already contain the mixture weights)
loss = loss_1 + loss_2



In [ ]:
#| notest
#from torchsurv.metrics.auc import Auc
n = 10

time = torch.randint(low=5, high=250, size=(n,), device='cpu').float()

event = torch.randint(low=0, high=2, size=(n,), device='cpu').bool()

estimate = torch.randn((n,), device='cpu')
eval_times = torch.tensor([30.], device='cpu')
auc = Auc(checks=False)

auc(estimate, event, time, auc_type = "cumulative", new_time=eval_times)

NameError: name 'Auc' is not defined

In [ ]:
#| export
class CrossEntropyLoss(nn.Module):
    """
    Cross entropy loss with ignore_index.
    """
    def __init__(self, ignore_index=-100, reduction='mean', weight=None, label_smoothing=0, soft_labels=False):
        super().__init__()
        self.ignore_index = ignore_index
        self.reduction = reduction
        self.label_smoothing = label_smoothing
        self.weight = weight
        if weight is not None:
            self.weight = weight.float()
        if soft_labels:
            # this has to be -100 for CE to work with soft labels
            ignore_index = -100
        self.soft_labels = soft_labels
        self.loss = nn.CrossEntropyLoss(ignore_index=ignore_index, reduction='none', weight=self.weight, label_smoothing=label_smoothing)
    
    def forward(self, x, y):
        """
        x: [bs x n classes x n patches]
        y: [bs x n patches] or [bs x n classes x n patches]
        """
        if x.is_nested:
            x = x.to_padded_tensor(padding=0)
        if y.is_nested:
            y = y.to_padded_tensor(padding=self.ignore_index)
        loss = self.loss(x, y)
        if self.soft_labels and y.dim() == 3:
            mask = y.sum(dim=1, keepdim=True) > 0  # [bs x 1 x soft_labels]
            loss = loss * mask
            loss = loss.sum() / mask.sum().clamp(min=1e-6) if self.reduction == 'mean' else loss.sum()
        else:
            mask = y != self.ignore_index
            loss = loss * mask
            loss = loss.sum() / mask.sum().clamp(min=1e-6) if self.reduction == 'mean' else loss.sum()
        return loss


In [ ]:
#| notest
criterion = CrossEntropyLoss(ignore_index=0, weight=torch.tensor([1,1,1,1,1]), reduction='mean', label_smoothing=0.)
batch_size = 10

n_patch = 721
n_class = 5
#m = torch.nn.Softmax(dim=-1)
logits = torch.randn(batch_size, n_class, n_patch)
target = torch.randint(0, n_class, size=(batch_size, n_patch))
criterion(logits, target), nn.CrossEntropyLoss(ignore_index=0, reduction='mean')(logits, target)

(tensor(1.9717), tensor(1.9717))

In [ ]:
#| notest
# soft labels
criterion = CrossEntropyLoss(ignore_index=100, weight=torch.tensor([1,1,1,1,1]), reduction='mean', label_smoothing=0., soft_labels=True)
logits = torch.randn(batch_size, n_class, n_patch)
target = torch.rand(batch_size, n_class, n_patch)
criterion(logits, target)


tensor(49.1424)

In [ ]:
#| notest
batch_size = 2
n_classes = 5
max_len = 480

# Create sequences of different lengths
seq_lens = torch.randint(50, max_len, (batch_size,))

# Create input tensors with different sequence lengths
x_list = [torch.randn(n_classes, length) for length in seq_lens]
y_list = [torch.randint(0, n_classes, size=(length,)) for length in seq_lens]
x_nested = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)
y_nested = torch.nested.as_nested_tensor(y_list, layout=torch.jagged)

criterion = CrossEntropyLoss(ignore_index=-100, weight=torch.tensor([1,1,1,1,1]), reduction='mean', label_smoothing=0., soft_labels=False)
criterion(x_nested, y_nested)

tensor(2.0479)

In [ ]:
#| export
class FocalLoss(nn.Module):
    """
    adapted from tsai, weighted multiclass focal loss
    https://github.com/timeseriesAI/tsai/blob/bdff96cc8c4c8ea55bc20d7cffd6a72e402f4cb2/tsai/losses.py#L116C1-L140C20
    """
    def __init__(self, 
                 weight=None, 
                 gamma=2., 
                 reduction='mean',
                 ignore_index=-100
                 ):
        super().__init__()
        self.weight = weight
        self.gamma = gamma
        self.reduction = reduction
        self.ignore_index = ignore_index
    
    __name__ = 'focalloss'
        
    def forward(self, x, y):
        """
        x: [bs x n classes x n patches]
        y: [bs x n patches]
        """
        bs = x.size(0)

        if y.dtype == torch.float32:
            y = y.long()

        if x.is_nested:
            x = x.to_padded_tensor(padding=0)
        if y.is_nested:
            y = y.to_padded_tensor(padding=self.ignore_index)

        if x.dim() == 3:
            x = x.permute(0,2,1) # bs x n_patches x n_classes
            x = x.reshape(-1, x.size(-1)) # bs * n_patches x n_classes

        log_prob = F.log_softmax(x, dim=-1)
        weight = self.weight.float().to(x.device) if self.weight is not None else None
        if y.dim() == 2:
            # hard labels
            y = y.flatten(start_dim=-2) # bs * num_patches
            valid_mask = y != self.ignore_index
            valid_y = y[valid_mask]
            valid_log_prob = log_prob[valid_mask]
            pt = valid_log_prob[torch.arange(len(valid_y)), valid_y].exp()
            ce = F.nll_loss(valid_log_prob, valid_y, weight=weight, reduction='none', ignore_index=self.ignore_index)
            loss = (1 - pt) ** self.gamma * ce
            loss = loss.sum() / valid_mask.sum().clamp(min=1e-5) if self.reduction == 'mean' else loss.sum()
        else:  # soft labels
            y = y.permute(0,2,1) # bs x n patches x n classes
            mask = (y.sum(dim=2) > 0).float() # bs x n patches
            mask = mask.flatten(start_dim=-2) # bs * n_patches
            y = y.reshape(-1, y.size(-1)) # bs * n_patches x n_classes

            if weight is not None:
                ce = -(y * log_prob * weight).sum(dim=-1) # bs * n_patches
            else:
                ce = -(y * log_prob).sum(dim=-1) # bs * n_patches
            pt = (y * log_prob.exp()).sum(dim=-1).clamp(min=1e-7, max=1)
            loss = (1 - pt) ** self.gamma * ce
            # Positions to ignore will have all zeros
            loss = loss * mask
            loss = loss.sum() / mask.sum().clamp(min=1e-5) if self.reduction == 'mean' else loss.sum()
        return loss

In [ ]:
#| notest
criterion = FocalLoss(gamma=0.7, weight=None, ignore_index=0)
batch_size = 10

n_patch = 721
n_class = 2
#m = torch.nn.Softmax(dim=-1)
logits = torch.randn(batch_size, n_class, n_patch)
target = torch.randint(0, n_class, size=(batch_size, n_patch))
criterion(logits, target.float())

tensor(0.7082)

In [ ]:
#| notest
# soft labels
criterion = FocalLoss(gamma=0.7, weight=torch.tensor([1,1,1,1,1]), ignore_index=0)
logits = torch.randn(batch_size, n_class, n_patch)
target = torch.rand(batch_size, n_class, n_patch)
criterion(logits, target)


tensor(2.8824)

In [ ]:
#| notest
batch_size = 2
n_classes = 5
max_len = 480

# Create sequences of different lengths
seq_lens = torch.randint(50, max_len, (batch_size,))

# Create input tensors with different sequence lengths
x_list = [torch.randn(n_classes, length) for length in seq_lens]
y_list = [torch.randint(0, n_classes, size=(length,)) for length in seq_lens]
y_list[0][0] = -100
x_nested = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)
y_nested = torch.nested.as_nested_tensor(y_list, layout=torch.jagged)
print(x_nested.shape, y_nested.shape)
criterion = FocalLoss(gamma=0.7, weight=torch.tensor([1,1,1,1,1]), ignore_index=-100)
criterion(x_nested, y_nested)



torch.Size([2, 5, j7]) torch.Size([2, j8])


tensor(1.7631)

In [ ]:
#| notest

batch_size = 2
n_classes = 5
max_len = 480

# Create sequences of different lengths
seq_lens = torch.randint(50, max_len, (batch_size,))

# Create input tensors with different sequence lengths
x_list = [torch.randn(n_classes, length) for length in seq_lens]
y_list = [torch.randn(n_classes, length) for length in seq_lens]
x_nested = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)
y_nested = torch.nested.as_nested_tensor(y_list, layout=torch.jagged)

criterion = FocalLoss(gamma=0.7, weight=None, ignore_index=0)
criterion(x_nested, y_nested)

tensor(2.2618)

In [ ]:
#| export
class Momentum(nn.Module):
    r"""
    adapted from torchsurv: https://github.com/Novartis/torchsurv/blob/main/src/torchsurv/loss/momentum.py
    Survival framework to momentum update learning to decouple batch size during model training.
    Two networks are concurrently trained, an online network and a target network. The online network outputs batches
    are concanetaed and used by the target network, so it virtually increase its batchsize.

    The target network (k)is updated using an exponential momentum average (**EMA**) using parameters from the online network (q).
    The online network is trained using a memory bank of previously computed log hazards, but only tracking loss from current batch.
    """

    def __init__(
        self,
        backbone: nn.Module,
        loss: Callable,
        batchsize: int = 16,
        steps: int = 10,
        rate: float = 0.999,
    ):
        """
        Initialise the momentum class. Use must provide their model as backbone.

        Args:
            backbone (nn.Module):
                Torch model to be use as backbone. The model must return either one (Cox) or two outputs (Weibull)
            loss (Callable): Torchsurv loss function (Cox, Weibull)
            batchsize (int, optional):
                Number of samples per batch. Defaults to 16.
            steps (int, optional):
                Number of queued batches to be stored for training. Defaults to 10.
            rate (float, optional):
                Exponential moving average rate. Defaults to 0.999.
        """
        super().__init__()
        self.rate = rate
        self.online = copy.deepcopy(backbone)  # q >> online network
        self.target = copy.deepcopy(backbone)  # k >> target network (momentum)
        self._init_encoder_k()
        self.loss = loss
        self.steps = steps

        # survival data structure
        self.survtuple = collections.namedtuple("survival", ["estimate", "event", "time"])

        # Hazards: current batch & memory
        self.memory_q = collections.deque(maxlen=batchsize)  # online (q)
        self.memory_k = collections.deque(maxlen=batchsize * steps)  # target (k)

    def forward(self, inputs: torch.Tensor, event: torch.Tensor, time: torch.Tensor) -> torch.Tensor:
        """Compute the loss for the current batch and update the memory bank using momentum class.

        Args:
            inputs (torch.Tensor): Input tensors to the backbone model.
            event (torch.Tensor, bool): Event indicator (= True if event occurred).
            time (torch.Tensor, float): Event or censoring time.

        Returns:
            torch.Tensor: A loss tensor for the current batch.
        """

        online_estimate = self.online(inputs)
        for estimate in zip(online_estimate, event, time):
            self.memory_q.append(self.survtuple(*estimate))
        loss = self._bank_loss()
        with torch.no_grad():
            self._update_momentum_encoder()
            target_estimate = self.target(inputs)
            for estimate in zip(target_estimate, event, time):
                self.memory_k.append(self.survtuple(*estimate))
        return loss

    @torch.no_grad()  # deactivates autograd
    def infer(self, inputs: torch.Tensor) -> torch.Tensor:
        """Evaluate data with target network

        Args:
            x (torch.Tensor): Input tensors to the backbone model

        Returns:
            torch.Tensor: Predictions from target (momentum) network without augmentation (.eval()) nor gradient.
        """
        self.target.eval()  # notify all your layers that you are in eval mode
        return self.target(inputs)

    def _bank_loss(self) -> torch.Tensor:
        """compute the negative log-likelihood from memory bank"""

        # Combine current batch and momentum
        if self.steps == 0:
            # normal training without momentum
            bank = self.memory_q
        else:
            bank = self.memory_k + self.memory_q
        assert all(x in bank[0]._fields for x in ["estimate", "event", "time"]), "All fields must be present"
        log_estimates = torch.stack([mem.estimate.cpu() for mem in bank]).squeeze()
        events = torch.stack([mem.event.cpu() for mem in bank]).squeeze()
        times = torch.stack([mem.time.cpu() for mem in bank]).squeeze()
        return self.loss(log_estimates, events, times)

    @torch.no_grad()
    def _update_momentum_encoder(self):
        """Exponential moving average"""
        for param_b, param_m in zip(self.online.parameters(), self.target.parameters()):
            param_m.data = param_m.data * self.rate + param_b.data * (1.0 - self.rate)

    @torch.no_grad()
    def _init_encoder_k(self):
        """
        Initialize the target network (encoder_k) with the parameters of the online network (encoder_q).
        The requires_grad attribute of the target network parameters is set to False to prevent gradient updates during training,
        ensuring that the target network remains a stable reference point.
        This method uses the `copy_` method to copy the parameters from the online network to the target network
        and sets the requires_grad attribute of the target network parameters to False to prevent gradient updates.
        """
        for param_q, param_k in zip(self.online.parameters(), self.target.parameters()):
            param_k.data.copy_(param_q.data)
            param_k.requires_grad = False

In [ ]:
#| export
class KLDivLoss(nn.Module):
    """
    Kullback-Leibler Divergence Loss with masking for ignore_index.
    Handles soft labels with ignore_index marked as -100.
    
    Args:
        logits: [bs x n_classes x pred_labels] - model predictions
        targets: [bs x n_classes x soft_labels] - soft labels, with ignore_index positions marked as 0
    """
    def __init__(self, reduction='mean'):
        super().__init__()
        self.kl_loss = nn.KLDivLoss(reduction='none')
        self.reduction = reduction
    def forward(self, logits, targets):
        bs = logits.size(0)
        if logits.is_nested:
            max_len = max(logits.size(-1) for logits in logits.unbind())
            logits = logits.to_padded_tensor(padding=0, output_size=(bs, logits.size(1), max_len))
            targets = targets.to_padded_tensor(padding=0, output_size=(bs, targets.size(1), max_len))
        # Create mask for valid positions (where target is not ignore_index)
        mask = targets.sum(dim=1, keepdim=True) > 0  # [bs x 1 x soft_labels]
        # Compute log probabilities
        log_probs = F.log_softmax(logits, dim=1)
        # Compute KL divergence loss
        loss = self.kl_loss(log_probs, targets)
        # Sum across class dimension and apply mask
        loss = (loss * mask).sum() / mask.sum().clamp(min=1e-6) if self.reduction == 'mean' else (loss * mask).sum()
        
        return loss

In [ ]:
#| notest
x = torch.randn(4,5,10)
y = torch.softmax(torch.randn(4,5,10), dim=1)

KLDivLoss()(x,y)


tensor(0.6321)

In [ ]:
#| notest

batch_size = 5
n_classes = 5
max_len = 480

# Create sequences of different lengths
seq_lens = torch.randint(50, max_len, (batch_size,))

# Create input tensors with different sequence lengths
x_list = [torch.randn(n_classes, length) for length in seq_lens]
y_list = [torch.softmax(torch.randn(n_classes, length), dim=0) for length in seq_lens]
x_nested = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)
y_nested = torch.nested.as_nested_tensor(y_list, layout=torch.jagged)

criterion = KLDivLoss()
criterion(x_nested, y_nested)

tensor(0.6608)

In [ ]:
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# import torch.distributed.nn

# from util import misc


# def compute_cross_entropy(p, q):
#     q = F.log_softmax(q, dim=-1)
#     loss = torch.sum(p * q, dim=-1)
#     return - loss.mean()


# def stablize_logits(logits):
#     logits_max, _ = torch.max(logits, dim=-1, keepdim=True)
#     logits = logits - logits_max.detach()
#     return logits


# @torch.no_grad()
# def concat_all_gather(tensor):
#     """
#     Performs all_gather operation on the provided tensors.
#     *** Warning ***: torch.distributed.all_gather has no gradient.
#     """
#     tensors_gather = [torch.ones_like(tensor)
#                       for _ in range(torch.distributed.get_world_size())]
#     torch.distributed.all_gather(tensors_gather, tensor, async_op=False)

#     output = torch.cat(tensors_gather, dim=0)
#     return output


# class MultiPosConLoss(nn.Module):
#     """
#     Multi-Positive Contrastive Loss: https://arxiv.org/pdf/2306.00984.pdf
#     """

#     def __init__(self, temperature=0.1):
#         super(MultiPosConLoss, self).__init__()
#         self.temperature = temperature
#         self.logits_mask = None
#         self.mask = None
#         self.last_local_batch_size = None

#     def forward(self, x, y):
#         feats = x    # feats shape: [B, D]
#         labels = y    # labels shape: [B]

#         feats = F.normalize(feats, dim=-1, p=2)
#         local_batch_size = feats.size(0)

#         all_feats = torch.cat(torch.distributed.nn.all_gather(feats), dim=0)
#         all_labels = concat_all_gather(labels)  # no gradient gather

#         # compute the mask based on labels
#         if local_batch_size != self.last_local_batch_size:
#             mask = torch.eq(labels.view(-1, 1),
#                             all_labels.contiguous().view(1, -1)).float().to(device)
#             self.logits_mask = torch.scatter(
#                 torch.ones_like(mask),
#                 1,
#                 torch.arange(mask.shape[0]).view(-1, 1).to(device) +
#                 local_batch_size * misc.get_rank(),
#                 0
#             )

#             self.last_local_batch_size = local_batch_size
#             self.mask = mask * self.logits_mask

#         mask = self.mask

#         # compute logits
#         logits = torch.matmul(feats, all_feats.T) / self.temperature
#         logits = logits - (1 - self.logits_mask) * 1e9

#         # optional: minus the largest logit to stablize logits
#         logits = stablize_logits(logits)

#         # compute ground-truth distribution
#         p = mask / mask.sum(1, keepdim=True).clamp(min=1.0)
#         loss = compute_cross_entropy(p, logits)

#         return loss

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()